In [12]:
import pandas as pd
import os
import glob

# 1. Cấu hình đường dẫn
clinical_csv_path = "../data/labels/Final_Matched_Clinical.csv" # Đường dẫn tới file CSV bạn vừa gửi
h5_dir = "../data/h5_features"        # Thư mục chứa các file .h5 đã trích xuất
output_csv_path = "../data/labels/labels.csv"

# 2. Đọc file clinical
df_clinical = pd.read_csv(clinical_csv_path)

# Tạo từ điển (dictionary) map Bệnh nhân -> Loại bệnh (Label)
# Bước này quy đổi text sang số: BLCA->0, BRCA->1, GBM->2, LGG->3
cohort_to_id = {'BLCA': 0, 'BRCA': 1, 'GBM': 2, 'LGG': 3}
stage_to_id = {float('nan'): "0", 'Stage I': "1", 
               'Stage IA': "2", 'Stage IB': "3",
               'Stage II': "4",
               'Stage IIA': "5", 'Stage IIB': "6",
               'Stage III': "7",'Stage IIIA': "8",
               'Stage IIIB': "9", 'Stage IIIC': "10",
               'Stage IV': "11", 'Stage X': "12"}
df_clinical['label'] = df_clinical['ajcc_pathologic_stage'].map(stage_to_id)

# Tạo một map dictionary để tra cứu nhanh: { 'TCGA-XF-AAN8': 0, ... }
patient_to_label = dict(zip(df_clinical['submitter_id'], df_clinical['label']))

# 3. Quét các file .h5 thực tế bạn đang có
h5_files = glob.glob(os.path.join(h5_dir, "*.h5"))

final_data = []

for h5_path in h5_files:
    # Lấy tên file, ví dụ: "TCGA-AR-A0TX-01Z-00-DX1.h5"
    filename = os.path.basename(h5_path)
    slide_id = filename.replace('.h5', '') # Bỏ đuôi .h5
    
    # CẮT LẤY 12 KÝ TỰ ĐẦU TIÊN ĐỂ TÌM BỆNH NHÂN
    patient_id = slide_id[:12] 
    
    # Kiểm tra xem bệnh nhân này có trong file clinical không
    if patient_id in patient_to_label:
        label = patient_to_label[patient_id]
        final_data.append({'slide_id': slide_id, 'label': label})
    else:
        print(f"Cảnh báo: Không tìm thấy thông tin lâm sàng cho file {slide_id}")

# 4. Lưu ra file labels.csv chuẩn cho Pipeline
df_final = pd.DataFrame(final_data)
os.makedirs(os.path.dirname(output_csv_path), exist_ok=True)
df_final.to_csv(output_csv_path, index=False)

print(f"Hoàn thành! Đã tạo file {output_csv_path} với {len(df_final)} slides hợp lệ.")
print("Mẫu dữ liệu:\n", df_final.head())

Hoàn thành! Đã tạo file ../data/labels/labels.csv với 30 slides hợp lệ.
Mẫu dữ liệu:
                                             slide_id label
0  TCGA-06-0210-01Z-00-DX4.58a33622-8ece-4934-b8b...     0
1  TCGA-B6-A0IH-01Z-00-DX1.3463B12E-D1B0-4FB2-B25...     8
2  TCGA-GU-A42P-01Z-00-DX1.3A4A8614-693F-452C-BB2...    11
3  TCGA-06-0210-01Z-00-DX2.1e456f3b-bebb-4606-907...     0
4  TCGA-D8-A1XC-01Z-00-DX1.E7494388-838B-4D35-9CA...     9


In [7]:
df_clinical['ajcc_pathologic_stage'].unique()
# df_clinical[df_clinical['ajcc_pathologic_stage']=='Stage X']

array(['Stage III', 'Stage IV', 'Stage I', 'Stage II', nan, 'Stage IIB',
       'Stage IIIA', 'Stage IIA', 'Stage IIIB', 'Stage IA', 'Stage IIIC',
       'Stage X', 'Stage IB'], dtype=object)

In [9]:
df_clinical[df_clinical['submitter_id'] == 'TCGA-06-0210']

,submitter_id,Study_Cohort,vital_status,vital_status_binary,OS_time_days,age_at_index,gender,ajcc_pathologic_stage,label
1470,TCGA-06-0210,GBMLGG,Dead,1.0,225,72,female,NaN,NaN


In [18]:
import torch

# Load the checkpoint
checkpoint = torch.load('../experiments/checkpoints/best_mil.pth', map_location=torch.device('cuda'))

print("type:",type(checkpoint))
print("keys:", list(checkpoint.keys()))

type: <class 'collections.OrderedDict'>
keys: ['v.0.weight', 'v.0.bias', 'u.0.weight', 'u.0.bias', 'w.weight', 'w.bias', 'classifier.weight', 'classifier.bias']
